In [13]:
# ------------------------------
# Load & Prepare Dataset (Year kept only for split/eval)
# ------------------------------
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

df = pd.read_csv("corn_dataset_final.csv")

# --- Config ---
TRAIN_YEARS = [2016, 2017, 2018, 2019]
TEST_YEARS  = [2020,2021,2022]
TARGET_COL  = "YIELD"

# Filter train/test years
train_df_all = df[df["year"].isin(TRAIN_YEARS)].copy()
test_df      = df[df["year"].isin(TEST_YEARS)].copy()

# ------------------------------
# Encode states 
# ------------------------------
states = sorted(train_df_all["state_name"].unique())
state_to_idx = {s: i for i, s in enumerate(states)}
train_df_all["state_idx"] = train_df_all["state_name"].map(state_to_idx)
test_df["state_idx"]      = test_df["state_name"].map(state_to_idx)

# ------------------------------
# Feature columns
# ------------------------------
# Drop year entirely from features
feature_cols = [
    "ppt (mm)_AVG", "tmax (degrees C)_AVG", "tmean (degrees C)_AVG", "tmin (degrees C)_AVG",  # climate
    "ssm_AVG", "susm_AVG",  # soil
    "EVI_AVG", "GCI_AVG", "NDWI_AVG", "NDVI_AVG",  # satellite
    "AWC", "SOM", "CEC", "state_idx",  # static # new historical average yield feature
]

# Columns to scale (all except target & year)
scaler_X = StandardScaler()
train_df_all[feature_cols] = scaler_X.fit_transform(train_df_all[feature_cols])
test_df[feature_cols]      = scaler_X.transform(test_df[feature_cols])

# Scale target
scaler_y = StandardScaler()
train_df_all["YIELD_STD"] = scaler_y.fit_transform(train_df_all[[TARGET_COL]])
test_df["YIELD_STD"]      = scaler_y.transform(test_df[[TARGET_COL]])

# ------------------------------
# Dataset class (flattened for single cross-attention)
# ------------------------------
class FlattenedCornDataset(torch.utils.data.Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)
        self.feature_cols = feature_cols
        self.target_col = "YIELD_STD"

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        X = torch.tensor(row[self.feature_cols].to_numpy(dtype=np.float32), dtype=torch.float32)
        y = torch.tensor(float(row[self.target_col]), dtype=torch.float32).unsqueeze(0)
        return X, y

# ------------------------------
# Create datasets & loaders
# ------------------------------
full_train_dataset = FlattenedCornDataset(train_df_all)
test_dataset       = FlattenedCornDataset(test_df)

train_idx, val_idx = train_test_split(
    range(len(full_train_dataset)),
    test_size=0.2,
    random_state=42
)

train_dataset = torch.utils.data.Subset(full_train_dataset, train_idx)
val_dataset   = torch.utils.data.Subset(full_train_dataset, val_idx)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32)
test_loader  = DataLoader(test_dataset, batch_size=32)

# ------------------------------
# Input info
# ------------------------------
sample_X, sample_y = train_dataset[0]

print(f"Flattened feature vector: {sample_X.shape[0]}")
print(f"Target shape: {sample_y.shape}")

Flattened feature vector: 14
Target shape: torch.Size([1])


In [14]:
class EarlyStopping:
    def __init__(self, patience=100, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float("inf")
        self.counter = 0
        self.stop = False

    def __call__(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1

        if self.counter >= self.patience:
            self.stop = True

In [15]:
import torch
import torch.nn as nn
import torchbnn as bnn

class BayesianYieldNN(nn.Module):
    def __init__(self, input_dim):
        super(BayesianYieldNN, self).__init__()
        
        # Prior parameters (as commonly used in literature)
        prior_mu = 0.0
        prior_sigma = 0.1
        
        self.model = nn.Sequential(
            bnn.BayesLinear(prior_mu, prior_sigma, input_dim, 128),
            nn.ReLU(),
            
            bnn.BayesLinear(prior_mu, prior_sigma, 128, 64),
            nn.ReLU(),
            
            bnn.BayesLinear(prior_mu, prior_sigma, 64, 1)
        )

    def forward(self, x):
        return self.model(x)
    


In [16]:
# %%
import torch.optim as optim
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

input_dim = len(feature_cols)

model = BayesianYieldNN(input_dim=input_dim).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

# KL divergence loss for Bayesian layers
kl_loss = bnn.BKLLoss(reduction='mean', last_layer_only=False)
kl_weight = 0.01  # IMPORTANT: tune this

early_stopper = EarlyStopping(patience=15, min_delta=0.001)

best_val_mse = float("inf")
best_model_path = "best_bnn_model.pth"

for epoch in range(200):
    model.train()
    train_preds, train_truths = [], []

    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()

        pred = model(X)

        mse = criterion(pred, y)
        kl  = kl_loss(model)

        loss = mse + kl_weight * kl
        loss.backward()
        optimizer.step()

        train_preds.append(pred.detach().cpu().numpy())
        train_truths.append(y.detach().cpu().numpy())

    train_preds = np.vstack(train_preds)
    train_truths = np.vstack(train_truths)

    train_mse = mean_squared_error(train_truths, train_preds)
    train_r2 = r2_score(train_truths, train_preds)

    train_preds_real = scaler_y.inverse_transform(train_preds)
    train_truths_real = scaler_y.inverse_transform(train_truths)
    train_r2_real = r2_score(train_truths_real, train_preds_real)

    # ------------------------------
    # Validation
    # ------------------------------
    model.eval()
    val_preds, val_truths = [], []

    with torch.no_grad():
        for X, y in val_loader:
            X, y = X.to(device), y.to(device)
            out = model(X)
            val_preds.append(out.cpu().numpy())
            val_truths.append(y.cpu().numpy())

    val_preds = np.vstack(val_preds)
    val_truths = np.vstack(val_truths)

    val_mse = mean_squared_error(val_truths, val_preds)
    val_r2 = r2_score(val_truths, val_preds)

    val_preds_real = scaler_y.inverse_transform(val_preds)
    val_truths_real = scaler_y.inverse_transform(val_truths)
    val_r2_real = r2_score(val_truths_real, val_preds_real)

    print(
        f"[Epoch {epoch+1}] "
        f"Train MSE(scaled): {train_mse:.4f}, Train R2(scaled): {train_r2:.4f}, Train R2(real): {train_r2_real:.4f} | "
        f"Val MSE(scaled): {val_mse:.4f}, Val R2(scaled): {val_r2:.4f}, Val R2(real): {val_r2_real:.4f}"
    )

    # Save best model
    if val_mse < best_val_mse:
        best_val_mse = val_mse
        torch.save(model.state_dict(), best_model_path)
        print(f"Saved new best BNN model at epoch {epoch+1} with Val MSE: {val_mse:.4f}")

    early_stopper(val_mse)
    if early_stopper.stop:
        print(f"Stopping early at epoch {epoch+1}")
        break


[Epoch 1] Train MSE(scaled): 0.9432, Train R2(scaled): 0.0582, Train R2(real): 0.0582 | Val MSE(scaled): 0.8413, Val R2(scaled): 0.1506, Val R2(real): 0.1506
Saved new best BNN model at epoch 1 with Val MSE: 0.8413
[Epoch 2] Train MSE(scaled): 0.7675, Train R2(scaled): 0.2336, Train R2(real): 0.2336 | Val MSE(scaled): 0.8106, Val R2(scaled): 0.1815, Val R2(real): 0.1815
Saved new best BNN model at epoch 2 with Val MSE: 0.8106
[Epoch 3] Train MSE(scaled): 0.7009, Train R2(scaled): 0.3002, Train R2(real): 0.3002 | Val MSE(scaled): 0.7615, Val R2(scaled): 0.2311, Val R2(real): 0.2311
Saved new best BNN model at epoch 3 with Val MSE: 0.7615
[Epoch 4] Train MSE(scaled): 0.6651, Train R2(scaled): 0.3359, Train R2(real): 0.3359 | Val MSE(scaled): 0.7483, Val R2(scaled): 0.2444, Val R2(real): 0.2444
Saved new best BNN model at epoch 4 with Val MSE: 0.7483
[Epoch 5] Train MSE(scaled): 0.6453, Train R2(scaled): 0.3557, Train R2(real): 0.3557 | Val MSE(scaled): 0.6984, Val R2(scaled): 0.2948, Val

In [17]:
# %%
from torch.utils.data import DataLoader

# ------------------------------
# Load Best BNN Model
# ------------------------------
model = BayesianYieldNN(input_dim=len(feature_cols)).to(device)
model.load_state_dict(torch.load("best_bnn_model.pth", map_location=device))
model.eval()

YEARS = [2020]

for year in YEARS:
    year_df = test_df[test_df["year"] == year].reset_index(drop=True)

    if len(year_df) == 0:
        print(f"No data for year {year}")
        continue

    year_dataset = FlattenedCornDataset(year_df)
    year_loader  = DataLoader(year_dataset, batch_size=32, shuffle=False)

    all_preds_scaled, all_truths_scaled = [], []

    with torch.no_grad():
        for X, y in year_loader:
            X, y = X.to(device), y.to(device)
            preds = model(X)
            all_preds_scaled.append(preds.cpu().numpy())
            all_truths_scaled.append(y.cpu().numpy())

    all_preds_scaled  = np.vstack(all_preds_scaled)
    all_truths_scaled = np.vstack(all_truths_scaled)

    all_preds_real  = scaler_y.inverse_transform(all_preds_scaled)
    all_truths_real = scaler_y.inverse_transform(all_truths_scaled)

    mse_scaled = mean_squared_error(all_truths_scaled, all_preds_scaled)
    r2_scaled  = r2_score(all_truths_scaled, all_preds_scaled)
    mse_real   = mean_squared_error(all_truths_real, all_preds_real)
    r2_real    = r2_score(all_truths_real, all_preds_real)

    print(f"--- Year {year} ---")
    print(f"MSE (scaled): {mse_scaled:.4f}, R2 (scaled): {r2_scaled:.4f}")
    print(f"MSE (real):   {mse_real:.4f}, R2 (real):   {r2_real:.4f}")


--- Year 2020 ---
MSE (scaled): 0.3888, R2 (scaled): 0.5604
MSE (real):   496.2614, R2 (real):   0.5604
